# ③ 设置账号并启动 iKataGo

先完成 `ikatago install.ipynb`。之后每次开机通常只需运行本文件。

首次输入用户名、密码；密码使用隐藏输入，不写入 Notebook 代码。已有账号默认保留，修改账号前先停止服务。分享 Notebook 前清空所有输出。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import importlib.util

STATE = Path.home() / ".config/katago-rtx50/notebook.json"
if not STATE.is_file():
    raise RuntimeError("请先运行 katago.ipynb 和 ikatago install.ipynb。")
WORK_DIR = Path(json.loads(STATE.read_text())["workDir"])
if not (WORK_DIR / "service.sh").is_file():
    raise RuntimeError("iKataGo 未安装，请运行 ikatago install.ipynb。")
helper_dir = str(WORK_DIR / "tools")
if helper_dir not in sys.path:
    sys.path.insert(0, helper_dir)
spec = importlib.util.spec_from_file_location("ikatago_notebook_service", WORK_DIR / "tools/ikatago_service.py")
manager = importlib.util.module_from_spec(spec)
spec.loader.exec_module(manager)


## 设置账号

In [ ]:
RESET_ACCOUNT = False  # 停服后改为 True 可重新设置账号
if RESET_ACCOUNT or not (WORK_DIR / "userlist.txt").is_file():
    import fcntl
    with (WORK_DIR / "service.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        if manager.active():
            raise RuntimeError("请先停止服务，再设置账号。")
        manager.configure()
else:
    print("保留已有账号。")


## 启动

客户端使用刚设置的用户名和密码；命令行客户端平台参数为 `--platform all`。进程运行不代表远程注册成功；连接失败可在云机私下查看 `logs/server.log`，不要公开该日志。


In [ ]:
if manager.active():
    print("服务已运行。")
else:
    subprocess.run([str(WORK_DIR / "service.sh"), "start"], check=True)
subprocess.run([str(WORK_DIR / "service.sh"), "status"], check=True)


## 停止（需要时手动启用）
关闭 Notebook 不会主动停止后台服务；关闭云实例会停止服务。

In [ ]:
STOP_SERVICE = False
if STOP_SERVICE:
    subprocess.run([str(WORK_DIR / "service.sh"), "stop"], check=True)
else:
    print("未停止服务。")
